In [5]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Administrateur\Documents\M2i\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [6]:
import pandas as pd
import numpy as np
import json
import time
import os

In [7]:
debug_mode = False

food_parquet = "../../data/food.parquet"
nutriment_parquet = "../../data/nutrients.parquet"
nutriscope_parquet = "../../data/nutriscope.parquet"
category_parquet = "../../data/category.parquet"

nutriments_list = ["fiber", "proteins", "energy", "saturated-fat", "sugars", "salt", "fruits-vegetables-legumes-estimate-from-ingredients"]
nutriments_list_light = ["fiber", "proteins", "energy", "saturated-fat", "sugars", "salt"]

sel_columns = ["nutriments", "code", "product_name", "nutriscore_grade", "brands"]


In [8]:
nutriscope_df = pd.read_parquet(nutriscope_parquet)

In [5]:
print(nutriscope_df.head(10))

            code nutriscore_grade                 brands  fiber  proteins  \
0  0000101209159                e                Bovetti    NaN      8.00   
1  0000105000417                b                 Lagg's    NaN      0.00   
2  0000111048403                b         Canola Harvest    NaN      0.00   
3  0000111301201                e         Canola Harvest   0.00      0.00   
4  0000111301263                d         Canola Harvest   0.00      0.00   
5  0000127534587                e    Today's Temptations   3.16      9.23   
6  0000204286484   not-applicable   allfitnessfactory.de    NaN     85.50   
7  0000209024937                e                           NaN      5.00   
8  0000234022960                e  La Fournée Campanière    NaN      8.60   
9  0000236555909                c         Wise Woodworks   2.90      8.82   

        energy  saturated-fat     sugars     salt  \
0  2524.000000      10.000000  32.000000  0.01000   
1          NaN            NaN        NaN  0.00

In [9]:
off_categories_tags_df = pd.read_parquet(food_parquet, columns=["code", "categories_tags"])
print(off_categories_tags_df.head(10))

            code                                    categories_tags
0  0000101209159  [en:breakfasts, en:spreads, en:sweet-spreads, ...
1  0000105000011                                          [en:null]
2  0000105000042  [en:plant-based-foods-and-beverages, en:bevera...
3  0000105000059  [en:beverages-and-beverages-preparations, en:p...
4  0000105000073                                               None
5  0000105000196                                          [en:null]
6  0000105000219                                          [en:null]
7  0000105000318                                          [en:null]
8  0000105000356  [en:plant-based-foods-and-beverages, en:bevera...
9  0000105000363                                          [en:null]


In [10]:
off_categories_tags_df[off_categories_tags_df["categories_tags"].apply(lambda x: isinstance(x, (list, np.ndarray)) and len(x) > 0)]

,code,categories_tags
0,0000101209159,"[en:breakfasts, en:spreads, en:sweet-spreads, ..."
1,0000105000011,[en:null]
2,0000105000042,"[en:plant-based-foods-and-beverages, en:bevera..."
3,0000105000059,"[en:beverages-and-beverages-preparations, en:p..."
5,0000105000196,[en:null]
...,...,...
4636436,5031915628654,[en:Dagligvarer]
4636437,5712878306089,[en:Dagligvarer]
4636438,9945616476098,[en:Dagligvarer]
4636439,5505688631718,[en:Dagligvarer]


In [11]:
cat_count = 20
exp = off_categories_tags_df[off_categories_tags_df["categories_tags"].apply(lambda x: isinstance(x, (list, np.ndarray)) and len(x) > 0)]["categories_tags"].explode()

datas = exp.value_counts()

tp2_df = pd.DataFrame({ "categorie": datas.head(cat_count).index, "nb_produits": datas.head(cat_count).values })

nb_nutriments = [6, 5, 4]

for category, count in datas.head(cat_count).items():
    codes = off_categories_tags_df[off_categories_tags_df["categories_tags"].apply(
        lambda x: category in x if isinstance(x, (list, np.ndarray)) else False
    )]["code"]

    print(category)
    for nu in nb_nutriments:
        tmp = nutriscope_df[(nutriscope_df["nb_nutriments"] == nu) & (nutriscope_df["code"].isin(codes))]

        tp2_df.loc[tp2_df["categorie"] == category, f"pct_{nu}"] = len(tmp) / len(codes) * 100

    tp2_df["completeness_score"] = (tp2_df["pct_6"] + tp2_df["pct_5"] * 5 / 6 + tp2_df["pct_4"] * 4 / 6)

tp2_df["produits_utiles"] = (tp2_df["nb_produits"] * tp2_df["completeness_score"] / 100)


en:plant-based-foods-and-beverages
en:plant-based-foods
en:snacks
en:sweet-snacks
en:beverages
en:dairies
en:cereals-and-potatoes
en:meats-and-their-products
en:fermented-foods
en:fermented-milk-products
en:fruits-and-vegetables-based-foods
en:condiments
en:beverages-and-beverages-preparations
en:meats
en:cereals-and-their-products
en:desserts
en:biscuits-and-cakes
en:meals
en:spreads
en:confectioneries


In [12]:
tp2_df.to_parquet(category_parquet, index=False)

In [13]:
print(tp2_df.sort_values("nb_produits", ascending=False))


                                  categorie  nb_produits      pct_6  \
0        en:plant-based-foods-and-beverages       551697  41.281899   
1                      en:plant-based-foods       481625  42.902466   
2                                 en:snacks       333294  36.423698   
3                           en:sweet-snacks       244973  36.540762   
4                              en:beverages       229465  20.741290   
5                                en:dairies       174657  32.365723   
6                   en:cereals-and-potatoes       166379  53.971956   
7               en:meats-and-their-products       148733  27.833097   
8                        en:fermented-foods       135490  32.357370   
9                en:fermented-milk-products       130582  32.245639   
10     en:fruits-and-vegetables-based-foods       124888  39.709980   
11                            en:condiments       122345  29.295844   
12  en:beverages-and-beverages-preparations       119165  22.080309   
13    